# A full business solution
## Business Challange:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits . 
We will be provided with company name and their primary website.

In [3]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from scraper import WebsiteScraper


In [4]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith('sk-proj-') and len(api_key) >10:
    print("API key is valid")
else:
    print("API key is invalid")
MODEL = "gpt-5-nano"
openai = OpenAI()

API key is valid


## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be the most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links":[
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of the links are most relevant to include in a brochure about the company,
    respond with full https URL in JSON format.
    Do not include Terms of Service, privacy policy, email links.

    Links (some might be relative links):

    """
    scraper = WebsiteScraper(url)
    links = scraper.get_links()
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://www.flightaware.com/live/"))


    Here is the list of links on the website https://www.flightaware.com/live/ -
    Please decide which of the links are most relevant to include in a brochure about the company,
    respond with full https URL in JSON format.
    Do not include Terms of Service, privacy policy, email links.

    Links (some might be relative links):

    /
/commercial/aeroapi/
/commercial/firehose/
/commercial/foresight/
/commercial/reports/rapid/
/commercial/reports/custom/
/commercial/integrated-maps/
/commercial/aviator/
/commercial/premium/
/commercial/global/
/commercial/fbo-toolbox/
/commercial/tv/
/commercial/globalbeacon/
https://www.flightaware.com/industry/airports
https://www.flightaware.com/industry/airlines
https://www.flightaware.com/industry/business
https://www.flightaware.com/industry/government
https://www.flightaware.com/industry/manufacturer
https://www.flightaware.com/industry/travel
/adsb/
/adsb/stats/
https://go.flightaware.com/skyawareanywhere
/adsb/coverage/
https://flightaw

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)  }
        ],
        response_format = {"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [9]:
select_relevant_links("https://www.flightaware.com/live/")

Selecting relevant links for https://www.flightaware.com/live/ by calling gpt-5-nano
Found 57 relevant links


{'links': [{'type': 'about page', 'url': 'https://www.flightaware.com/about/'},
  {'type': 'careers page',
   'url': 'https://www.flightaware.com/about/careers/'},
  {'type': 'history page',
   'url': 'https://www.flightaware.com/about/history/'},
  {'type': 'datasources page',
   'url': 'https://www.flightaware.com/about/datasources/'},
  {'type': 'industry page',
   'url': 'https://www.flightaware.com/industry/airports'},
  {'type': 'industry page',
   'url': 'https://www.flightaware.com/industry/airlines'},
  {'type': 'industry page',
   'url': 'https://www.flightaware.com/industry/business'},
  {'type': 'industry page',
   'url': 'https://www.flightaware.com/industry/government'},
  {'type': 'industry page',
   'url': 'https://www.flightaware.com/industry/manufacturer'},
  {'type': 'industry page',
   'url': 'https://www.flightaware.com/industry/travel'},
  {'type': 'commercial product page',
   'url': 'https://www.flightaware.com/commercial/aeroapi/'},
  {'type': 'commercial produ

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [10]:
def get_page_content(url):
    return WebsiteScraper(url).get_content()

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = get_page_content(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links: \n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += get_page_content(url)
    return result


In [12]:
print(fetch_page_and_all_relevant_links("https://www.flightaware.com/live/"))

Selecting relevant links for https://www.flightaware.com/live/ by calling gpt-5-nano
Found 35 relevant links
## Landing Page:

Live Flight Tracker - FlightAware

Products
Data Products
AeroAPI
Flight data API with on-demand flight status and flight tracking data.
FlightAware Firehose
Streaming flight data feed for enterprise integrations with real-time, historical and predictive flight data.
FlightAware Foresight
Predictive technology to strengthen customer trust in your operations
Rapid Reports
Quickly purchase historical reports delivered via email.
Custom Reports
Consultative detailed and customized flight tracking data reports.
Integrated Mapping Solutions
Incorporate FlightAware maps in your web and mobile applications
Applications
FlightAware Aviator
The ultimate flight tracking suite for small aircraft/general aviation (GA) owners and operators.
Premium Subscriptions
A personalized flight-following experience with unlimited alerts and more.
FlightAware Global
The industry standa

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [14]:
get_brochure_user_prompt("FlightAware", "https://www.flightaware.com/live/")

Selecting relevant links for https://www.flightaware.com/live/ by calling gpt-5-nano
Found 22 relevant links


'\nYou are looking at a company called: FlightAware\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nLive Flight Tracker - FlightAware\n\nProducts\nData Products\nAeroAPI\nFlight data API with on-demand flight status and flight tracking data.\nFlightAware Firehose\nStreaming flight data feed for enterprise integrations with real-time, historical and predictive flight data.\nFlightAware Foresight\nPredictive technology to strengthen customer trust in your operations\nRapid Reports\nQuickly purchase historical reports delivered via email.\nCustom Reports\nConsultative detailed and customized flight tracking data reports.\nIntegrated Mapping Solutions\nIncorporate FlightAware maps in your web and mobile applications\nApplications\nFlightAware Aviator\nThe ultimate flight tracking suite for small aircraft/general aviation (GA) owners and operators.\nP

In [15]:
from IPython.display import Markdown


def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [16]:
create_brochure("FlightAware", "https://www.flightaware.com/live/")

Selecting relevant links for https://www.flightaware.com/live/ by calling gpt-5-nano
Found 39 relevant links


# FlightAware Company Brochure

---

## About FlightAware

FlightAware is a leading global provider of live flight tracking and aviation data products. They deliver real-time, predictive, and historical flight information to a diverse clientele, including airports, airlines, business aviation, government agencies, manufacturers, and the travel industry. FlightAware supports millions of users worldwide ranging from individual pilots and aviation enthusiasts to enterprise-level aviation operations.

---

## Products & Services

### Data Products
- **AeroAPI**: A robust flight data API providing on-demand flight statuses and tracking data.
- **FlightAware Firehose**: A streaming flight data feed for enterprise customers that includes real-time, historical, and predictive flight data.
- **FlightAware Foresight**: Industry-leading predictive technology designed to enhance customer trust and operational performance.
- **Rapid Reports**: Quick purchase of historical aviation reports conveniently delivered via email.
- **Custom Reports**: Tailored, consultative flight tracking data reports built to meet specific client needs.
- **Integrated Mapping Solutions**: Tools to incorporate FlightAware’s maps into business web and mobile applications.

### Applications
- **FlightAware Aviator**: The comprehensive flight tracking suite tailored specifically for small aircraft and general aviation pilots.
- **Premium Subscriptions**: Personalized experiences with unlimited flight alerts and enhanced tracking features.
- **FlightAware Global**: The gold standard platform trusted by business aviation owners and operators for seamless flight tracking.
- **FlightAware FBO Toolbox**: An all-in-one tool to improve Fixed Base Operator (FBO) operations and drive sales growth.
- **FlightAware TV**: Large full-screen flight tracking maps ideal for operators and FBOs.
- **GlobalBeacon**: A GADSS-compliant global tracking and alerting solution for airlines and aircraft operators.

---

## Industries Served

- **Airports**: Optimizing operations and enhancing passenger experience through real-time data.
- **Airlines**: Reliable flight status and predictive analytics supporting scheduling and customer service.
- **Business Aviation & General Aviation**: Flight tracking and operational support tailored for private and small aircraft.
- **Government**: Critical flight data for safety, monitoring, and logistical purposes.
- **Manufacturers**: Access flight operations data to support aviation hardware lifecycle.
- **Travel Industry**: FlightAware enhances travel planning and disruption management with accurate flight status data.

---

## Community & Engagement

FlightAware fosters a strong global community of aviation enthusiasts and professionals through:
- Aviation photo sharing and community tagging.
- Discussion forums and current "Squawks" (pilot and user reported irregularities).
- Popular aviation maps like the **MiseryMap** for tracking delays and cancellations.
- Educational resources including blogs, engineering insights, and webinars.

---

## Careers at FlightAware

FlightAware offers exciting career opportunities for those passionate about aviation and aviation technology. The company culture emphasizes innovation, accuracy, and improving global air travel transparency and safety. Employees work in a dynamic environment that values expertise in software development, data science, aviation operations, and customer support.

For current job openings and career development information, prospective candidates can visit FlightAware’s careers page.

---

## Why Choose FlightAware?

- **Real-Time Precision**: Up-to-the-second flight tracking leveraging a global network of ADS-B receivers and data sources.
- **Comprehensive Solutions**: From APIs and custom reporting to predictive analytics, FlightAware services meet various operational needs.
- **Trusted by Industry Leaders**: Serving a broad customer base including major airlines, business aviation users, manufacturers, and travel companies.
- **Innovative Predictive Technology**: FlightAware Foresight enhances operational reliability and customer confidence.
- **Global Coverage**: Worldwide flight tracking capability compliant with modern aviation tracking regulations (e.g., GADSS).

---

## Get in Touch

- **Explore more:** [FlightAware Website](https://flightaware.com)
- **Contact Sales & Support:** Available via the website’s contact page.
- **Download the App:** Access live flight tracking from anywhere via the official FlightAware mobile app.

---

FlightAware — *Your Trusted Partner in Flight Tracking and Aviation Data Solutions*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI, **but this time in German**,
with the familiar typewriter animation

In [17]:
from IPython.display import update_display

brochure_system_prompt_german = """
You are a marketing assistant.

Generate the brochure entirely in German.
Use natural, professional German.
Keep headings, bullet points, and formatting in German.
"""

def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt_german},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [18]:
stream_brochure("FlightAware", "https://www.flightaware.com/live/")

Selecting relevant links for https://www.flightaware.com/live/ by calling gpt-5-nano
Found 27 relevant links


ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))